# Act 1 — Sub‑act A: PVDAQ-only EDA

EDA is exploration-first:
- learn the data (schema, units, time index, cadence, missingness)
- surface emerging patterns
- summarize insights + open questions at the end

This notebook focuses on **PVDAQ-only**.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.pvdaq import compute_facility_stats, load_all_raw_csvs
from src.viz.plots import set_plot_style

raw_dir = ROOT / "data" / "raw"
interim_daily = ROOT / "data" / "interim" / "pvdaq_11797_daily.csv"

if interim_daily.exists():
    daily = pd.read_csv(interim_daily)
    df = daily.rename(columns={"date": "_ts"}).copy()
    df["_ts"] = pd.to_datetime(df["_ts"], errors="coerce")
    df["_system_id"] = 11797
else:
    frames = load_all_raw_csvs(raw_dir)
    if not frames:
        raise FileNotFoundError(f"Add PVDAQ CSV files to {raw_dir} (see data/raw/README.md)")
    df = next(iter(frames.values()))

# Metric candidates
metric_mean = [c for c in df.columns if c.endswith("_daily_mean")]
metric_sum = [c for c in df.columns if c.endswith("_daily_sum")]
metric_mean, metric_sum

## Stage 1 — Snapshot tables (coverage, cadence, missingness)


In [ ]:
set_plot_style()

# Choose primary columns for EDA (we'll compare both)
mean_col = metric_mean[0] if metric_mean else None
sum_col = metric_sum[0] if metric_sum else None

stats = compute_facility_stats(
    df.rename(columns={"_ts": "_ts", "_system_id": "_system_id"}).assign(
        _target=pd.to_numeric(df[mean_col], errors="coerce") if mean_col else np.nan
    )
)

# Expected daily index
full_days = pd.date_range(df["_ts"].min().floor("D"), df["_ts"].max().floor("D"), freq="D")
observed_days = pd.to_datetime(df["_ts"]).dt.floor("D")
missing_days = full_days.difference(pd.DatetimeIndex(observed_days.unique()).sort_values())

snapshot = pd.DataFrame(
    {
        "system_id": [df["_system_id"].iloc[0]],
        "start": [df["_ts"].min()],
        "end": [df["_ts"].max()],
        "rows": [len(df)],
        "median_interval_minutes": [stats.median_interval_minutes],
        "missing_days": [len(missing_days)],
        "pct_missing_days": [float(len(missing_days) / len(full_days) * 100.0)],
        "pct_missing_mean_metric": [float(pd.to_numeric(df[mean_col], errors="coerce").isna().mean() * 100.0) if mean_col else np.nan],
        "pct_missing_sum_metric": [float(pd.to_numeric(df[sum_col], errors="coerce").isna().mean() * 100.0) if sum_col else np.nan],
    }
)

snapshot

## Stage 2 — Sanity plots (time series + distributions)


In [ ]:
mean_series = pd.to_numeric(df[mean_col], errors="coerce") if mean_col else None
sum_series = pd.to_numeric(df[sum_col], errors="coerce") if sum_col else None

fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)

if mean_col:
    axes[0, 0].plot(df["_ts"], mean_series, linewidth=1)
    axes[0, 0].set_title("Daily mean power (proxy)")
    axes[0, 1].hist(mean_series.dropna(), bins=60)
    axes[0, 1].set_title("Distribution: daily mean power")
else:
    axes[0, 0].set_title("No *_daily_mean column found")

if sum_col:
    axes[1, 0].plot(df["_ts"], sum_series, linewidth=1)
    axes[1, 0].set_title("Daily energy sum")
    axes[1, 1].hist(sum_series.dropna(), bins=60)
    axes[1, 1].set_title("Distribution: daily energy")
else:
    axes[1, 0].set_title("No *_daily_sum column found")

out_path = ROOT / "reports" / "figures" / "pvdaq_11797_stage2_sanity.png"
fig.savefig(out_path, dpi=150)
out_path

## Stage 3 — Data issues (gaps, outliers)


In [ ]:
# Largest missing-day spans
if len(missing_days) > 0:
    gaps = pd.DataFrame({"missing_day": missing_days})
    gaps["gap_group"] = (gaps["missing_day"].diff() != pd.Timedelta(days=1)).cumsum()
    gap_spans = (
        gaps.groupby("gap_group")["missing_day"]
        .agg(gap_start="min", gap_end="max", n_days="count")
        .sort_values("n_days", ascending=False)
        .reset_index(drop=True)
    )
else:
    gap_spans = pd.DataFrame(columns=["gap_start", "gap_end", "n_days"])

gap_spans.head(10)

## Stage 4 — PVDAQ insights recap (fill in after exploration)

- **What we learned**:
  - 
- **Open questions**:
  - 
- **Risks / caveats**:
  - 
